## Step 1: Setup and Imports

In [ ]:
# Import required libraries
import sys
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../configs/.env')

# Add src to path
sys.path.append(str(Path('../src').resolve()))

from neo4j_agent import Neo4jAgent

print("✅ Imports successful")

## Step 2: Connect to Neo4j

In [ ]:
# Get credentials from environment
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

# Initialize Neo4j agent
agent = Neo4jAgent(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD
)

print(f"✅ Connected to Neo4j at {NEO4J_URI}")

## Step 3: Ingest Sample Project

In [ ]:
# Define project paths
project_path = Path('../examples/sample_project')
build_xml_path = project_path / 'build.xml'

# Ingest project
print("🔄 Starting ingestion...")
report = agent.ingest_project(
    version="v1.0.0",
    project_path=project_path,
    build_xml_path=str(build_xml_path)
)

# Display results
print("\n✅ Ingestion Complete!")
print(f"   Classes: {report['classes_ingested']}")
print(f"   Methods: {report['methods_extracted']}")
print(f"   Dependencies: {report['dependencies_mapped']}")
print(f"   Documentation Links: {report['documentation_links']}")
print(f"   Time: {report['execution_time']:.2f}s")

## Step 4: Query the Knowledge Graph

In [ ]:
# Query 1: List all classes
print("📊 Query 1: All Classes\n")

query = """
MATCH (c:Class)
RETURN c.name AS ClassName, c.category AS Category, c.gitTag AS Version
ORDER BY c.name
"""

results = agent.query(query)
for record in results:
    print(f"  • {record['ClassName']} ({record['Category']}) - {record['Version']}")

In [ ]:
# Query 2: Find class methods
print("📊 Query 2: Methods in ExampleTestMethod\n")

query = """
MATCH (c:Class {name: 'ExampleTestMethod'})-[:DEFINES_METHOD]->(m:Method)
RETURN m.name AS MethodName, m.visibility AS Visibility, m.returnType AS ReturnType
ORDER BY m.name
"""

results = agent.query(query)
for record in results:
    print(f"  • {record['Visibility']} {record['ReturnType']} {record['MethodName']}()")

In [ ]:
# Query 3: Find inheritance hierarchy
print("📊 Query 3: Inheritance Hierarchy\n")

query = """
MATCH path = (child:Class)-[:EXTENDS]->(parent:Class)
RETURN child.name AS Child, parent.name AS Parent
"""

results = agent.query(query)
for record in results:
    print(f"  {record['Child']} → {record['Parent']}")

In [ ]:
# Query 4: Node statistics
print("📊 Query 4: Database Statistics\n")

query = """
MATCH (n)
RETURN labels(n)[0] AS NodeType, count(*) AS Count
ORDER BY Count DESC
"""

results = agent.query(query)
for record in results:
    print(f"  {record['NodeType']}: {record['Count']}")

## Step 5: Visualize Results with Pandas

In [ ]:
import pandas as pd

# Get all classes with their categories
query = """
MATCH (c:Class)
RETURN c.name AS Class, c.category AS Category, c.visibility AS Visibility
ORDER BY c.category, c.name
"""

results = agent.query(query)
df = pd.DataFrame(results)

print("\n📊 Classes DataFrame:")
print(df)

print("\n📈 Category Distribution:")
print(df['Category'].value_counts())

## Step 6: Cleanup

In [ ]:
# Close Neo4j connection
agent.close()
print("✅ Neo4j connection closed")

## Next Steps

Try these exercises:

1. **Find Dependencies:**
   ```cypher
   MATCH (c:Class)-[:DEPENDS_ON]->(dep:ExternalDependency)
   RETURN c.name, dep.name
   ```

2. **Documentation Links:**
   ```cypher
   MATCH (md:MarkdownFile)-[:RESOLVES_TO]->(c:Class)
   RETURN md.path, c.name
   ```

3. **Method Count by Class:**
   ```cypher
   MATCH (c:Class)-[:DEFINES_METHOD]->(m:Method)
   RETURN c.name, count(m) AS MethodCount
   ORDER BY MethodCount DESC
   ```

**Resources:**
- [Neo4j Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)
- [DEVELOPER_GUIDE.md](../docs/DEVELOPER_GUIDE.md)
- [WALKTHROUGH.md](../WALKTHROUGH.md)